In [ ]:
# ============================================================
#  SOUND THERAPY: TONE GENERATOR + VOICE + PDF (Colab ready)
# ============================================================
#
#  How to use
#    1. Set the options in this cell.
#    2. Run all cells (Runtime > Run all).
#    3. The WAV (and PDF) play/show inline and download in Colab.
#       In Jupyter the files are saved next to the notebook.
#
#  Presets
#    Pure tone + math PDF ........ VOICE "none", INCLUDE_RESEARCH_NOTES False
#    528 Hz + female voice + PDF . VOICE "female", INCLUDE_RESEARCH_NOTES True
#    528 Hz + male voice + PDF ... VOICE "male",   INCLUDE_RESEARCH_NOTES True
#    WAV only .................... MAKE_PDF False
# ============================================================

# ---- Audio ----
DURATION_SEC  = 120       # length of the audio in seconds
SAMPLE_RATE   = 44100     # samples per second (44100 = CD quality)
MASTER_VOLUME = 0.25      # peak loudness after mixing, 0.0 to 1.0
APPLY_FADE    = True      # short fade in/out to prevent clicks

TONE_MODE = "tones"       # "tones" or "binaural"

# TONE_MODE "tones": which tones play, and their frequencies in Hz.
# Left/Right play in that ear only; Center plays in both ears.
USE_LEFT_TONE   = False
USE_RIGHT_TONE  = False
USE_CENTER_TONE = True
LEFT_FREQ_HZ    = 528
RIGHT_FREQ_HZ   = 432
CENTER_FREQ_HZ  = 528

# TONE_MODE "binaural": one carrier tone, split +/- BEAT_HZ/2 between
# the ears, so the brain hears a beat at BEAT_HZ.
BEAT_HZ    = 3.0          # perceived beat (2 to 4 Hz = delta/theta range)
CARRIER_HZ = 200.0        # base frequency heard in both ears (150 to 400 typical)

# ---- Voice ----
VOICE = "none"            # "none", "female" (gTTS), or "male" (eSpeak NG)
AFFIRMATION_TEXT     = "You're beautiful!"
AFFIRMATION_INTERVAL = 30.0   # seconds between repeats
VOICE_VOLUME         = 0.40   # voice level relative to the tone (tone = 1.0)

# ---- PDF ----
MAKE_PDF  = True
PDF_TITLE = ""            # your own title, or "" for an automatic one
                          # (for example "528 Hz Pure Tone")
INCLUDE_RESEARCH_NOTES = False   # adds the 528 Hz research paragraph and references
EQUATION_IMAGES        = False   # True: typeset equations from Codecogs (needs internet)
                                 # False: equations as text (works offline)

# ---- Output ----
FILE_PREFIX   = "tone_mix"   # WAV and PDF names start with this; a timestamp is added
PLAY_INLINE   = True         # show the audio player and the PDF in the notebook
AUTO_DOWNLOAD = True         # download the files (Colab only)

In [ ]:
# ============================================================
#  CODE (nothing to edit here; the options are in the cell above)
# ============================================================
import importlib
import io
import shutil
import subprocess
import sys
from datetime import datetime


# ---- Packages: install only what the chosen options need ----
def ensure_package(pip_name, import_name=None):
    """Install a package with pip if it cannot be imported."""
    try:
        importlib.import_module(import_name or pip_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])


ensure_package("numpy")
ensure_package("scipy")
ensure_package("matplotlib")
ensure_package("reportlab")
if VOICE == "female":
    ensure_package("gTTS", "gtts")
if EQUATION_IMAGES:
    ensure_package("requests")

import numpy as np
import matplotlib
matplotlib.use("Agg")  # draw to memory; no display needed
import matplotlib.pyplot as plt
from scipy.io import wavfile
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas
from reportlab.lib.utils import ImageReader
from IPython.display import Audio, IFrame, display

# Colab is detected once, for the auto-download feature.
try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    colab_files = None
    IN_COLAB = False


# ============================================================
#  1. SETTINGS CHECK
# ============================================================
def documented_tone():
    """
    Which tone the PDF describes.

    Returns a dict with:
        mode  : "pure" or "binaural"
        freq  : the frequency plotted (carrier frequency in binaural mode)
        title : the automatic PDF title
        beat, carrier : present in binaural mode only
    In "tones" mode the priority is center, then left, then right.
    """
    if TONE_MODE == "binaural":
        return {"mode": "binaural", "freq": CARRIER_HZ, "beat": BEAT_HZ, "carrier": CARRIER_HZ,
                "title": f"{BEAT_HZ:g} Hz Binaural Beat (Carrier {CARRIER_HZ:g} Hz)"}
    if USE_CENTER_TONE:
        freq = CENTER_FREQ_HZ
    elif USE_LEFT_TONE:
        freq = LEFT_FREQ_HZ
    elif USE_RIGHT_TONE:
        freq = RIGHT_FREQ_HZ
    else:
        freq = CENTER_FREQ_HZ  # nothing is on; the PDF still documents the center setting
    return {"mode": "pure", "freq": freq, "title": f"{freq:g} Hz Pure Tone"}


def validate_config():
    """
    Check the settings for common mistakes.

    Raises ValueError with a plain message for an error that stops the run.
    Returns a list of warnings for things that run but may not be intended.
    """
    warnings = []

    if TONE_MODE not in ("tones", "binaural"):
        raise ValueError(f'TONE_MODE must be "tones" or "binaural", got {TONE_MODE!r}')
    if VOICE not in ("none", "female", "male"):
        raise ValueError(f'VOICE must be "none", "female", or "male", got {VOICE!r}')
    if DURATION_SEC <= 0:
        raise ValueError(f"DURATION_SEC must be positive, got {DURATION_SEC}")
    if SAMPLE_RATE <= 0:
        raise ValueError(f"SAMPLE_RATE must be positive, got {SAMPLE_RATE}")
    if not (0.0 <= MASTER_VOLUME <= 1.0):
        raise ValueError(f"MASTER_VOLUME must be between 0.0 and 1.0, got {MASTER_VOLUME}. "
                         "Values above 1.0 would cause clipping distortion.")

    for name, freq in [("LEFT_FREQ_HZ", LEFT_FREQ_HZ), ("RIGHT_FREQ_HZ", RIGHT_FREQ_HZ),
                       ("CENTER_FREQ_HZ", CENTER_FREQ_HZ), ("CARRIER_HZ", CARRIER_HZ)]:
        if freq <= 0:
            raise ValueError(f"{name} must be positive, got {freq}")
        if freq > SAMPLE_RATE / 2:
            warnings.append(f"{name} = {freq} Hz is above the Nyquist frequency "
                            f"({SAMPLE_RATE / 2:g} Hz); the tone will alias.")

    if TONE_MODE == "tones" and not (USE_LEFT_TONE or USE_RIGHT_TONE or USE_CENTER_TONE):
        warnings.append("All tone toggles are off; the output will be silence"
                        + (" apart from the voice." if VOICE != "none" else "."))
    if TONE_MODE == "binaural" and BEAT_HZ <= 0:
        raise ValueError(f"BEAT_HZ must be positive, got {BEAT_HZ}")

    if VOICE != "none":
        if not AFFIRMATION_TEXT.strip():
            raise ValueError("AFFIRMATION_TEXT is empty")
        if AFFIRMATION_INTERVAL <= 0:
            raise ValueError(f"AFFIRMATION_INTERVAL must be positive, got {AFFIRMATION_INTERVAL}")
        if VOICE_VOLUME < 0:
            raise ValueError(f"VOICE_VOLUME must be 0 or more, got {VOICE_VOLUME}")

    if MAKE_PDF and INCLUDE_RESEARCH_NOTES and documented_tone()["freq"] != 528:
        warnings.append("INCLUDE_RESEARCH_NOTES is True, but the PDF documents "
                        f"{documented_tone()['freq']:g} Hz; the notes describe 528 Hz research.")
    return warnings


def print_config_summary(wav_filename, pdf_filename):
    """Print what this run will make."""
    print("=" * 56)
    print("CONFIGURATION SUMMARY")
    print("=" * 56)
    print(f"Duration:       {DURATION_SEC} seconds")
    print(f"Sample rate:    {SAMPLE_RATE} Hz")
    print(f"Master volume:  {MASTER_VOLUME}")
    print(f"Fade in/out:    {'Yes' if APPLY_FADE else 'No'}")
    print("-" * 56)
    if TONE_MODE == "binaural":
        print("Mode:           Binaural beat")
        print(f"Carrier:        {CARRIER_HZ:g} Hz")
        print(f"Beat:           {BEAT_HZ:g} Hz")
        print(f"Left ear:       {CARRIER_HZ - BEAT_HZ / 2:.2f} Hz")
        print(f"Right ear:      {CARRIER_HZ + BEAT_HZ / 2:.2f} Hz")
    else:
        print("Mode:           Tones")
        tones = []
        if USE_LEFT_TONE:
            tones.append(f"Left ({LEFT_FREQ_HZ:g} Hz)")
        if USE_RIGHT_TONE:
            tones.append(f"Right ({RIGHT_FREQ_HZ:g} Hz)")
        if USE_CENTER_TONE:
            tones.append(f"Center ({CENTER_FREQ_HZ:g} Hz)")
        print(f"Active tones:   {', '.join(tones) if tones else 'NONE (silence)'}")
    print("-" * 56)
    if VOICE == "none":
        print("Voice:          None")
    else:
        engine = "female (gTTS)" if VOICE == "female" else "male (eSpeak NG)"
        print(f"Voice:          {engine}")
        print(f"Says:           {AFFIRMATION_TEXT!r} every {AFFIRMATION_INTERVAL:g} seconds")
        print(f"Voice volume:   {VOICE_VOLUME}")
    print("-" * 56)
    if MAKE_PDF:
        print(f"PDF title:      {PDF_TITLE or documented_tone()['title']}")
        print(f"Research notes: {'Yes' if INCLUDE_RESEARCH_NOTES else 'No'}")
        print(f"Equations:      {'Typeset images (Codecogs)' if EQUATION_IMAGES else 'Text'}")
    else:
        print("PDF:            Off")
    print("-" * 56)
    print(f"WAV output:     {wav_filename}")
    if MAKE_PDF:
        print(f"PDF output:     {pdf_filename}")
    print("=" * 56 + "\n")


# ============================================================
#  2. TONES
# ============================================================
def sine_wave(freq_hz, n_samples, sr):
    """
    A pure sine wave: n_samples of float64 values in [-1.0, 1.0].

    endpoint=False keeps the wave periodic if it is looped: with
    endpoint=True the last sample would repeat the first one.
    """
    t = np.linspace(0.0, n_samples / sr, n_samples, endpoint=False)
    return np.sin(2.0 * np.pi * freq_hz * t)


def apply_fade_io(wave, sr, fade_ms=20):
    """Return a copy of wave with a linear fade-in and fade-out of fade_ms."""
    result = wave.copy()
    n = len(result)
    k = int(sr * (fade_ms / 1000.0))
    if k == 0 or k * 2 >= n:
        return result
    result[:k] *= np.linspace(0.0, 1.0, k)
    result[-k:] *= np.linspace(1.0, 0.0, k)
    return result


def normalize_stereo(left, right, target_peak):
    """Scale both channels together so the loudest sample equals target_peak."""
    peak = max(np.abs(left).max(), np.abs(right).max(), 1e-12)  # 1e-12 avoids dividing by zero
    scale = target_peak / peak
    return left * scale, right * scale


def build_tones(n_samples, sr):
    """
    The tone part of the mix, as (left, right) float64 arrays with peak 1.0
    per tone. Which tones play depends on TONE_MODE and the toggles.
    """
    left = np.zeros(n_samples, dtype=np.float64)
    right = np.zeros(n_samples, dtype=np.float64)
    if TONE_MODE == "binaural":
        # Example: carrier 200 Hz, beat 3 Hz: left 198.5 Hz, right 201.5 Hz
        left += sine_wave(max(0.1, CARRIER_HZ - BEAT_HZ / 2.0), n_samples, sr)
        right += sine_wave(max(0.1, CARRIER_HZ + BEAT_HZ / 2.0), n_samples, sr)
    else:
        if USE_LEFT_TONE:
            left += sine_wave(LEFT_FREQ_HZ, n_samples, sr)
        if USE_RIGHT_TONE:
            right += sine_wave(RIGHT_FREQ_HZ, n_samples, sr)
        if USE_CENTER_TONE:
            center = sine_wave(CENTER_FREQ_HZ, n_samples, sr)
            left += center
            right += center
    return left, right


# ============================================================
#  3. VOICE
# ============================================================
def load_audio_mono(path, sr):
    """
    Read any audio file (MP3 or WAV) as mono float32 samples at sample
    rate sr, scaled so the loudest sample is 1.0. Decoding uses ffmpeg,
    which Colab has; on a Mac install it with: brew install ffmpeg
    """
    if shutil.which("ffmpeg") is None:
        raise RuntimeError("ffmpeg is needed to decode the voice clip. "
                           "Colab has it; on a Mac run: brew install ffmpeg")
    resampled = "affirmation_resampled.wav"
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", path,
                    "-ac", "1", "-ar", str(sr), "-sample_fmt", "s16", resampled], check=True)
    _, samples = wavfile.read(resampled)
    samples = samples.astype(np.float32)
    peak = float(np.max(np.abs(samples))) if len(samples) else 0.0
    return samples / peak if peak > 0 else samples


def make_voice_clip(voice, text, sr):
    """
    Synthesize the affirmation once and return it as mono float32 samples
    at sample rate sr, peak 1.0.
        "female": gTTS (Google text-to-speech, needs internet, no API key)
        "male"  : eSpeak NG, voice en+m3 at speed 135
    """
    if voice == "female":
        from gtts import gTTS
        gTTS(text=text, lang="en", slow=False).save("affirmation.mp3")
        return load_audio_mono("affirmation.mp3", sr)

    if shutil.which("espeak-ng") is None and shutil.which("apt-get") is not None:
        subprocess.run("apt-get install -y espeak-ng > /dev/null 2>&1", shell=True)  # Colab
    if shutil.which("espeak-ng") is None:
        raise RuntimeError("eSpeak NG is not installed. Colab installs it automatically; "
                           "on a Mac run: brew install espeak-ng")
    subprocess.run(["espeak-ng", "-v", "en+m3", "-s", "135", "-w", "affirmation.wav", text], check=True)
    return load_audio_mono("affirmation.wav", sr)


def build_voice_track(clip, n_samples, sr, interval_sec):
    """
    A mono track of n_samples with the clip placed at time 0 and then every
    interval_sec seconds. A repeat that would run past the end is cut off.
    """
    track = np.zeros(n_samples, dtype=np.float32)
    step = max(1, int(sr * interval_sec))
    for start in range(0, n_samples, step):
        length = min(len(clip), n_samples - start)
        track[start:start + length] = clip[:length]
    return track


# ============================================================
#  4. MIX
# ============================================================
def build_audio():
    """
    The finished stereo audio as a float32 array of shape (n_samples, 2):
    tones, plus the voice track in both ears if a voice is on, normalized
    to MASTER_VOLUME, with the fade applied last.
    """
    n_samples = int(SAMPLE_RATE * DURATION_SEC)
    left, right = build_tones(n_samples, SAMPLE_RATE)

    if VOICE != "none":
        clip = make_voice_clip(VOICE, AFFIRMATION_TEXT, SAMPLE_RATE)
        voice = VOICE_VOLUME * build_voice_track(clip, n_samples, SAMPLE_RATE, AFFIRMATION_INTERVAL)
        left = left + voice
        right = right + voice

    left, right = normalize_stereo(left, right, MASTER_VOLUME)
    if APPLY_FADE:
        left = apply_fade_io(left, SAMPLE_RATE)
        right = apply_fade_io(right, SAMPLE_RATE)
    return np.column_stack([left, right]).astype(np.float32)


# ============================================================
#  5. PDF
# ============================================================
RESEARCH_LEAD_IN = [
    "Research suggests the 528 Hz audio frequency may influence stress-",
    "related biomarkers, including lowering cortisol, the body's primary",
    "stress hormone, and increasing oxytocin, the body's primary hormone",
    "for bonding, trust, and social connection.",
]
RESEARCH_REFERENCES = [
    "Fukushima et al. (2018). Effects of 528 Hz on endocrine and mood states.",
    "URL: https://www.ncbi.nlm.nih.gov/pmc/articles/PMC5871151/",
    "",
    "Hirokawa et al. (2015). Rat stress markers under 528 Hz.",
    "URL: https://www.jstage.jst.go.jp/article/jvms/77/7/77_150015/_article",
]


def latex_image(latex):
    """Fetch a typeset equation from Codecogs as an image (needs internet)."""
    import requests
    url = "https://latex.codecogs.com/png.image?\\dpi{150} " + latex
    response = requests.get(url, timeout=20)
    response.raise_for_status()
    return ImageReader(io.BytesIO(response.content))


def waveform_image(freq, amplitude, sr):
    """A dark-red plot of the first 5 ms of the tone, as an image for the PDF."""
    window = 0.005
    t = np.linspace(0.0, window, int(sr * window), endpoint=False)
    fig, ax = plt.subplots(figsize=(6, 2))
    ax.plot(t * 1000.0, amplitude * np.sin(2.0 * np.pi * freq * t), color="#8B0000", linewidth=2)
    ax.set_title(f"{freq:g} Hz Waveform (first 5 ms)", fontsize=12)
    ax.set_xlabel("Time (ms)")
    ax.set_ylabel("Amplitude")
    ax.grid(True, alpha=0.3)
    buf = io.BytesIO()
    plt.savefig(buf, format="png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    return ImageReader(buf)


def draw_equation(c, y, latex, text, image_width):
    """
    Draw one equation at height y: a typeset image when EQUATION_IMAGES is
    True and the fetch works, otherwise the text version in Courier.
    Returns the next free y.
    """
    if EQUATION_IMAGES:
        try:
            img = latex_image(latex)
            pixel_w, pixel_h = img.getSize()
            image_height = image_width * pixel_h / pixel_w
            c.drawImage(img, 80, y - image_height + 8, width=image_width, height=image_height, mask="auto")
            return y - image_height - 4
        except Exception as e:
            print(f"  (Typeset equation unavailable, using text instead: {e})")
    c.setFont("Courier", 10)
    c.drawString(80, y, text)
    return y - 18


def make_pdf(filename, amplitude, sr):
    """
    A one-page PDF describing the tone: title, optional research notes,
    the general sine-wave equation, the variable definitions, the worked
    equation(s) for this run, the waveform plot, optional references,
    and a footer with the creation time.
    """
    tone = documented_tone()
    title = PDF_TITLE.strip() or tone["title"]
    freq = tone["freq"]

    c = canvas.Canvas(filename, pagesize=letter)
    width, height = letter
    y = height - 60

    c.setFont("Helvetica-Bold", 18)
    c.drawString(60, y, title)
    y -= 34

    if INCLUDE_RESEARCH_NOTES:
        c.setFont("Helvetica", 10)
        for line in RESEARCH_LEAD_IN:
            c.drawString(60, y, line)
            y -= 14
        y -= 10

    c.setFont("Helvetica-Bold", 12)
    c.drawString(60, y, "Equation:")
    y -= 16
    c.setFont("Helvetica", 10)
    c.drawString(60, y, "The general equation for a pure tone (one single frequency) is defined as:")
    y -= 16
    y = draw_equation(c, y, "s[n] = A\\sin(2\\pi f \\frac{n}{F_s})",
                      "s[n] = A x sin(2 pi f n / Fs)", image_width=150)

    y -= 4
    c.setFont("Helvetica", 10)
    c.drawString(60, y, "Where:")
    y -= 16
    for line in [f"A  = amplitude (peak loudness, scale 0 to 1) = {amplitude}",
                 "f  = frequency in Hz (cycles per second)",
                 "n  = sample index (0, 1, 2, ...)",
                 f"Fs = sample rate = {sr} samples per second"]:
        c.drawString(80, y, line)
        y -= 14
    y -= 8

    c.setFont("Helvetica-Bold", 12)
    c.drawString(60, y, "Worked example:" if tone["mode"] == "pure" else "Worked example (left and right ears):")
    y -= 18
    if tone["mode"] == "pure":
        y = draw_equation(c, y, f"s[n] = {amplitude:.2f}\\sin(2\\pi \\cdot {freq:g} \\cdot n / {sr})",
                          f"s[n] = {amplitude:.2f} x sin(2 pi x {freq:g} x n / {sr})", image_width=210)
    else:
        fl = tone["carrier"] - tone["beat"] / 2.0
        fr = tone["carrier"] + tone["beat"] / 2.0
        y = draw_equation(c, y, f"s_L[n] = {amplitude:.2f}\\sin(2\\pi \\cdot {fl:.2f} \\cdot n / {sr})",
                          f"Left:  s_L[n] = {amplitude:.2f} x sin(2 pi x {fl:.2f} x n / {sr})", image_width=230)
        y = draw_equation(c, y, f"s_R[n] = {amplitude:.2f}\\sin(2\\pi \\cdot {fr:.2f} \\cdot n / {sr})",
                          f"Right: s_R[n] = {amplitude:.2f} x sin(2 pi x {fr:.2f} x n / {sr})", image_width=230)
    y -= 8

    c.setFont("Helvetica-Bold", 12)
    c.drawString(60, y, "Waveform visualization:")
    y -= 10
    c.drawImage(waveform_image(freq, amplitude, sr), 60, y - 120, width=400, height=120)
    y -= 138

    if INCLUDE_RESEARCH_NOTES:
        c.setFont("Helvetica-Bold", 12)
        c.drawString(60, y, "References:")
        y -= 16
        c.setFont("Helvetica", 9)
        for line in RESEARCH_REFERENCES:
            c.drawString(60, y, line)
            y -= 12
        y -= 6

    c.setFont("Helvetica-Oblique", 9)
    c.drawString(60, y, "Generated digitally using the sampled sine-wave equation above.")
    y -= 14
    c.setFont("Helvetica", 8)
    c.drawString(60, y, f"Created: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    c.save()

In [ ]:
# ============================================================
#  RUN
# ============================================================
try:
    for warning in validate_config():
        print("WARNING:", warning)
except ValueError as e:
    print(f"SETTINGS ERROR: {e}")
    raise SystemExit(1)

STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
WAV_FILENAME = f"{FILE_PREFIX}_{STAMP}.wav"
PDF_FILENAME = f"{FILE_PREFIX}_info_{STAMP}.pdf"
print_config_summary(WAV_FILENAME, PDF_FILENAME)

# Audio: build, save as 16-bit WAV, play, download
print("Generating audio...")
audio = build_audio()
wavfile.write(WAV_FILENAME, SAMPLE_RATE, np.int16(np.clip(audio, -1.0, 1.0) * 32767))
print(f"WAV saved: {WAV_FILENAME}")

if PLAY_INLINE:
    try:
        display(Audio(WAV_FILENAME))
    except Exception as e:
        print(f"  (Inline audio player unavailable: {e})")
if AUTO_DOWNLOAD and IN_COLAB:
    try:
        colab_files.download(WAV_FILENAME)
    except Exception as e:
        print(f"  (WAV download failed; use the file browser on the left. {e})")

# PDF: build, show, download
if MAKE_PDF:
    print("Generating PDF...")
    make_pdf(PDF_FILENAME, MASTER_VOLUME, SAMPLE_RATE)
    print(f"PDF saved: {PDF_FILENAME}")
    if PLAY_INLINE:
        try:
            display(IFrame(PDF_FILENAME, width=600, height=800))
        except Exception as e:
            print(f"  (Inline PDF view unavailable; open it from the file browser. {e})")
    if AUTO_DOWNLOAD and IN_COLAB:
        try:
            colab_files.download(PDF_FILENAME)
        except Exception as e:
            print(f"  (PDF download failed; use the file browser on the left. {e})")

print("\nDone.")